In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names

# 🚀 데이터셋 정보: Falah/military_machinery_prompts
# 📜 의미: 군사 기계(군용 차량, 장비 등)와 관련된 AI 프롬프트 예시 데이터셋입니다.
# ✨ 설명: 이 데이터셋은 다양한 '프롬프트(prompts)' 문자열을 담고 있습니다. 우리는 이 텍스트 데이터가 어떻게 AI 모델의 입력을 구성하는지 학습하며, 단순히 데이터를 보는 것을 넘어 '더 좋은 프롬프트를 만들어내는' 창의적인 AI의 역할을 시뮬레이션해 볼 것입니다!

# --- 설정 상수 ---
DATASET_NAME = "Falah/military_machinery_prompts"
SPLIT_NAME = "train"
SAMPLE_COUNT = 10 # 분석을 위해 상위 10개 샘플만 사용합니다.

# ===============================================================================
# 🌟 튜터의 코멘트: 데이터 로딩 단계 (성능과 안정성을 고려한 마법!)
# ===============================================================================
print("🤖 튜터가 데이터셋을 불러옵니다. 스트리밍(Streaming) 모드를 먼저 시도할게요!")

dataset = None
try:
    # 1. 스트리밍 모드 (Streaming)로 로드 시도 (가장 빠르고 메모리 효율적)
    # streaming=True는 데이터가 필요할 때마다 조금씩 가져오게 합니다.
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✅ 스트리밍 모드 로드 성공! 아주 가볍고 빠릅니다!")
    
except Exception as e:
    # 2. 스트리밍 로드 실패 시 (네트워크 문제 등), 일반 모드로 작은 테스트 샘플을 강제 다운로드
    print(f"⚠️ 스트리밍 로드 중 오류 발생 ({e}). 일반 다운로드 모드로 전환하여 테스트할게요.")
    try:
        # 일반 모드 (streaming=False)로 로드하되, 최소한의 샘플만 사용합니다.
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
        print("✅ 일반 다운로드 모드 로드 성공! 이제 안전하게 진행할 수 있어요.")
    except Exception as e_fail:
        print(f"❌ 데이터셋 로드 자체에 실패했습니다. 데이터셋 이름({DATASET_NAME})을 확인해 주세요. 에러: {e_fail}")
        exit()


# ===============================================================================
# 🧺 데이터 샘플 준비
# ===============================================================================
# 전체 데이터를 다 돌면 시간이 너무 오래 걸리니까, 상위 SAMPLE_COUNT 개만 가져와서 빠르게 체험해 봅시다.
# Dataset 객체에서 take() 메서드는 스트리밍/일반 모드 모두에서 안전하게 샘플을 가져올 수 있습니다.

# ⭐️ 핵심 패턴: take()를 사용하여 샘플 이터레이터를 만듭니다.
sample_iterator = dataset.take(SAMPLE_COUNT)
# ⭐️ list()로 한번에 샘플들을 메모리에 올려 처리합니다. (가장 빠릅니다!)
sample_data_list = list(sample_iterator)

print(f"\n✨ 총 {len(sample_data_list)}개의 샘플을 사용하여 실습을 진행합니다!")

# ===============================================================================
# 💡 미션 1: 데이터 구조 확인 및 기초 통계 분석 (정량적 학습)
# ===============================================================================
print("\n" + "="*70)
print("🚀 미션 1: 데이터 살펴보기와 길이 측정하기 (Basic Inspection)")
print("="*70)

# sample_data_list는 리스트 형태이므로, 일반적인 리스트 루프를 사용합니다.
print("▶︎ [샘플 구조 확인]: 첫 번째 샘플의 구조를 확인해 봅시다.")
first_sample = sample_data_list[0]
print(f"  - 샘플 타입: {type(first_sample)}")
print(f"  - 사용 가능한 필드 이름: {list(first_sample.keys())}")

# 📊 정량적 분석: 샘플들이 가지고 있는 정보의 평균적인 길이는 어느 정도일까요?
total_prompt_length = 0
for i, sample in enumerate(sample_data_list):
    if 'prompts' in sample:
        length = len(sample['prompts'])
        total_prompt_length += length
        print(f"  - Sample {i+1} Prompt Length: {length} characters.")

average_length = total_prompt_length / len(sample_data_list)
print("-" * 30)
print(f"📊 [통계 분석 결과]: {len(sample_data_list)}개 샘플의 평균 프롬프트 길이는 약 {average_length:.2f}자 입니다.")


# ===============================================================================
# 🤖 미션 2: AI 프롬프트 엔지니어링 실습 (창의적 사용 예시)
# ===============================================================================
print("\n" + "="*70)
print("🧠 미션 2: 프롬프트 품질 향상기 (Prompt Quality Enhancer)")
print("="*70)
print("✨ 목표: 간단한 원본 프롬프트를 AI가 처리하기 좋은 '구조화된 지시문'으로 확장해 봅시다.")

def enhance_prompt(original_prompt: str) -> str:
    """
    원본 프롬프트에 역할(Role), 배경(Context), 출력 형식(Format)을 추가하여
    더 전문적이고 구조화된 AI 명령어로 변환하는 함수입니다.
    """
    # 1. 역할 부여 (Role Assignment): AI에게 역할을 부여하면 성능이 올라갑니다.
    role = "당신은 군사 장비 전문가이자 전문 스토리텔러입니다."
    
    # 2. 배경 설정 (Context Setting): 상황을 명확히 알려줍니다.
    context = "다음 원본 프롬프트를 기반으로, 미래 전쟁 시나리오를 위한 웅장한 서사시적 문장을 묘사해야 합니다."
    
    # 3. 출력 형식 강제 (Format Enforcement): 결과물의 형태를 지정합니다.
    format_instruction = "\n[출력 형식]: 반드시 <역할>:..., <맥락>:, <최종 출력>:... 태그를 사용하여 작성하고, 전체 길이는 200자를 넘기지 마세요."
    
    # 최종 구조화된 지시문 조합
    enhanced = f"""
--- 지시사항 시작 ---
{role}
{context}

[원본 입력]: "{original_prompt}"

{format_instruction}
--- 지시사항 끝 ---
"""
    return enhanced.strip()

# 💻 실습 실행: 샘플 데이터에 함수 적용
print("\n▶︎ [실행]: 상위 샘플들을 구조화된 프롬프트로 업그레이드해 봅시다!")
print("--- 이 과정을 통해 AI 모델은 요청사항을 놓치지 않고, 더 깊이 있는 답변을 생성할 수 있습니다. ---")

for i, sample in enumerate(sample_data_list):
    if 'prompts' in sample and sample['prompts']:
        original = sample['prompts']
        enhanced = enhance_prompt(original)
        
        print(f"\n{'='*30} [Sample {i+1} 결과]")
        print(f"  [원본 (Raw)]:\n    --> {original[:60]}...")
        print(f"  [업그레이드된 프롬프트]:\n{enhanced}")

print("\n\n🎉 축하합니다! 데이터를 로드하고, 통계를 분석하고, 심지어 AI 프롬프트 엔지니어링까지 성공적으로 수행했습니다!")
print("이처럼 데이터셋은 단순한 자료가 아니라, AI를 학습시키고 사용자가 AI를 더 잘 활용하는 방법(프롬프트)을 배우는 최고의 교재입니다!")